# STEP4 - 결과 확인 및 통합 비교표

STEP2/3(원격)에서 나온 `sbert_<name>.json` 3개 + `sbert_raw_stats.json`을 읽어서,
1주차 PLM 5개 + 2주차 전통 ML 4개 결과와 합쳐 12개 baseline 통합 비교표를 만들고,
`sbert_summary.txt`를 작성한다. 여기서는 새로운 계산(F1 등)을 하지 않고 이미 만들어진
결과 파일들을 정리/비교만 한다.

실행 전: 원격에서 만든 `train_results/sbert_*.json`, `sbert_raw_stats.json`을
이 프로젝트의 `3주차/train_results/`로 복사해뒀는지 확인.

In [ ]:
import sys
import json
import glob
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import common
from common import TARGET_SYMPTOMS, RESULTS_DIR, WEEK1_RESULTS_DIR, WEEK2_RESULTS_DIR, KLUE_ROBERTA_VAL_MACRO_F1


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

In [ ]:
sbert_results = {}
for name in common.SBERT_MODELS:
    path = RESULTS_DIR / f"sbert_{name}.json"
    if path.exists():
        sbert_results[name] = load_json(path)
    else:
        print(f"[경고] {path} 없음 - STEP3 결과를 옮겨왔는지 확인하세요")

raw_stats = {}
if common.RAW_STATS_PATH.exists():
    raw_stats = load_json(common.RAW_STATS_PATH)
else:
    print(f"[경고] {common.RAW_STATS_PATH} 없음 - STEP2/3 실행 결과를 옮겨왔는지 확인하세요")

sbert_results

In [ ]:
week1_jsons = sorted(glob.glob(str(WEEK1_RESULTS_DIR / "step5_*.json")))
week2_jsons = sorted(glob.glob(str(WEEK2_RESULTS_DIR / "stepD_*.json")))

week1_results = {Path(p).stem: load_json(p) for p in week1_jsons}
week2_results = {Path(p).stem: load_json(p) for p in week2_jsons}

print(f"1주차 PLM 결과 {len(week1_results)}개, 2주차 전통ML 결과 {len(week2_results)}개 로드")

In [ ]:
rows = []

for name, r in week1_results.items():
    row = {"category": "PLM(full fine-tuning)", "name": r.get("name", name),
           "hf_id": r.get("hf_id", ""), "val_macro_f1": r.get("val_macro_f1")}
    for s in TARGET_SYMPTOMS:
        row[f"f1_{s}"] = r.get(f"f1_{s}")
    rows.append(row)

for name, r in week2_results.items():
    row = {"category": "전통 ML", "name": r.get("name", name),
           "hf_id": f"{r.get('vectorizer', '')}+{r.get('model', '')}", "val_macro_f1": r.get("val_macro_f1")}
    for s in TARGET_SYMPTOMS:
        row[f"f1_{s}"] = r.get(f"f1_{s}")
    rows.append(row)

for name, r in sbert_results.items():
    row = {"category": "SBERT(frozen)+Linear", "name": r.get("name", name),
           "hf_id": r.get("hf_id", ""), "val_macro_f1": r.get("val_macro_f1")}
    for s in TARGET_SYMPTOMS:
        row[f"f1_{s}"] = r.get(f"f1_{s}")
    rows.append(row)

comparison_df = pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
comparison_df

In [ ]:
gap_lines = []
for name, r in sbert_results.items():
    f1 = r.get("val_macro_f1")
    if f1 is None:
        continue
    gap = KLUE_ROBERTA_VAL_MACRO_F1 - f1
    gap_lines.append(
        f"{name} ({r.get('hf_id')}): val_macro_f1={f1:.4f}, "
        f"klue-roberta(0.6023, 같은 뼈대 full fine-tuning) 대비 격차={gap:+.4f}"
    )

for line in gap_lines:
    print(line)

In [ ]:
if common.SPLIT_CHECK_PATH.exists():
    split_check_text = common.SPLIT_CHECK_PATH.read_text(encoding="utf-8")
else:
    split_check_text = "(train_val_split_check.txt 없음 - STEP1을 먼저 실행하세요)"

print(split_check_text[:800])

In [ ]:
summary_lines = []
summary_lines.append("=== 3주차 SBERT baseline 결과 요약 ===")
summary_lines.append("")
summary_lines.append("[데이터 검증 - STEP1 결과]")
summary_lines.append(split_check_text)
summary_lines.append("")

for name, hf_id in common.SBERT_MODELS.items():
    summary_lines.append(f"--- 모델: {name} ({hf_id}) ---")
    r = sbert_results.get(name)
    s = raw_stats.get(name, {})

    if r is None:
        summary_lines.append("  [경고] 결과 없음 - STEP3 미실행 또는 결과 파일 미전달")
        summary_lines.append("")
        continue

    status_line = f"  status: {r.get('status')}"
    if r.get("error"):
        status_line += f" / error: {r.get('error')}"
    summary_lines.append(status_line)

    summary_lines.append(f"  max_length: {s.get('max_length', common.MAX_LENGTH)}")
    summary_lines.append(f"  벡터 shape: train={s.get('shape_train')}, val={s.get('shape_val')}")
    summary_lines.append(f"  인코딩 소요시간: train={s.get('encode_time_train_s')}s, val={s.get('encode_time_val_s')}s")
    summary_lines.append(f"  분류기 학습 소요시간: {r.get('train_time_s')}s")
    summary_lines.append(
        f"  분류기 설정: lr={s.get('classifier_lr')}, epochs={s.get('classifier_epochs')}, "
        f"batch_size={s.get('classifier_batch_size')}, optimizer={s.get('classifier_optimizer')}, "
        f"seed={s.get('seed')}, threshold={s.get('threshold')}"
    )
    summary_lines.append(f"  256 토큰 초과(truncation) 건수: train={s.get('truncated_train')}, val={s.get('truncated_val')}")
    summary_lines.append(f"  val Macro-F1: {r.get('val_macro_f1')}")
    for sym in TARGET_SYMPTOMS:
        summary_lines.append(f"    f1_{sym}: {r.get(f'f1_{sym}')}")

    if r.get("val_macro_f1") is not None:
        gap = KLUE_ROBERTA_VAL_MACRO_F1 - r["val_macro_f1"]
        summary_lines.append(f"  klue-roberta(0.6023, 같은 뼈대 full fine-tuning) 대비 격차: {gap:+.4f}")
    else:
        summary_lines.append("  klue-roberta 대비 격차: 계산 불가")
    summary_lines.append("")

summary_lines.append("[방법론 한계 명시]")
summary_lines.append("- dev set을 별도로 분리하지 않음: train으로 학습, val로 평가 (2주차와 동일 방식).")
summary_lines.append("- threshold는 9개 증상 전부 0.5로 고정 (val 기준 별도 튜닝 없음).")
summary_lines.append("- 분류기는 PLM과 공정 비교를 위해 Linear(768->9) 한 층만 사용, MLP 등은 포함하지 않음.")
summary_lines.append("")

summary_lines.append("[12개 baseline 통합 비교 - val Macro-F1 내림차순]")
summary_lines.append(comparison_df[["category", "name", "val_macro_f1", "f1_오심"]].to_string(index=False))

summary_text = "
".join(summary_lines)
with open(common.SUMMARY_PATH, "w", encoding="utf-8") as f:
    f.write(summary_text)

print(f"저장 완료 -> {common.SUMMARY_PATH}")
print(summary_text)

In [ ]:
comparison_csv_path = RESULTS_DIR / "baseline_comparison_all.csv"
comparison_df.to_csv(comparison_csv_path, index=False, encoding="utf-8-sig")
print(f"통합 비교표 저장 -> {comparison_csv_path}")
comparison_df